# Text Embeddings

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../Datasets/customer_support_tickets.csv")
print(data.shape)

(8469, 17)


In [3]:
data["Text"] = (data["Ticket Subject"].fillna("")
    + " "
    + data["Ticket Description"].fillna("")
)

In [4]:
import re
def clean_text(text):
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Keep letters, spaces, and apostrophes
    text = re.sub(r"[^a-z\s']", " ", text)
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text
data["Clean_Text"] = data["Text"].apply(clean_text)

#### Load the embedding model

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2",device="cpu")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### Test One Sentence

In [6]:
sample_text = data["Clean_Text"].iloc[0]
embedding = model.encode(
    sample_text,
    convert_to_numpy=True
)
print("Embedding shape:", embedding.shape)

Embedding shape: (384,)


#### Generate Embedding

In [7]:
ticket_texts = data["Clean_Text"].tolist()

In [8]:
ticket_embeddings = model.encode(
    ticket_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/265 [00:00<?, ?it/s]

In [9]:
import numpy as np
np.save("../Datasets/processed/ticket_embeddings.npy",ticket_embeddings)

In [10]:
data[
    [
        "Ticket ID",
        "Ticket Type",
        "Ticket Subject",
        "Ticket Description",
        "Clean_Text"
    ]
].to_csv("../Datasets/processed/ticket_metadata.csv",index=False)

## FAISS Vector Search

In [11]:
import faiss
import numpy as np
import pandas as pd

In [12]:
print("FAISS version:", faiss.__version__)

FAISS version: 1.15.0


In [13]:
ticket_embeddings = np.load(
    "../Datasets/processed/ticket_embeddings.npy"
).astype("float32")

In [14]:
print("Embedding shape:", ticket_embeddings.shape)

Embedding shape: (8469, 384)


In [15]:
normalize_embeddings=True
dimension = ticket_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(ticket_embeddings)
print("Number of vectors in FAISS:", index.ntotal)
len(ticket_embeddings)

Number of vectors in FAISS: 8469


8469

In [16]:
faiss.write_index(
    index,
    "../Datasets/processed/ticket_faiss.index"
)

#### Test semantic search

In [17]:
query = "My payment was charged but my account still shows unpaid."

In [18]:
query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

In [19]:
print(query_embedding.shape)

(1, 384)


In [20]:
k = 5

scores, indices = index.search(
    query_embedding,
    k
)

In [21]:
print("Indices:", indices)
print("Scores:", scores)

Indices: [[2701  876  973 5769  926]]
Scores: [[0.5922682  0.59140253 0.5912138  0.5813449  0.58025444]]


In [22]:
ticket_metadata = pd.read_csv(
    "../Datasets/processed/ticket_metadata.csv"
)

In [23]:
results = ticket_metadata.iloc[indices[0]].copy()
results["Similarity"] = scores[0]
results["Rank"] = range(1, len(results) + 1)
results = results[
    [
        "Rank",
        "Ticket ID",
        "Ticket Type",
        "Ticket Subject",
        "Ticket Description",
        "Similarity"
    ]
]

results

,Rank,Ticket ID,Ticket Type,Ticket Subject,Ticket Description,Similarity
2701,1,2702,Product inquiry,Payment issue,I'm facing a problem with my {product_purchase...,0.592268
876,2,877,Technical issue,Payment issue,I'm having an issue with the {product_purchase...,0.591403
973,3,974,Billing inquiry,Payment issue,I'm having an issue with the {product_purchase...,0.591214
5769,4,5770,Cancellation request,Payment issue,I've forgotten my password for my {product_pur...,0.581345
926,5,927,Billing inquiry,Payment issue,I'm having an issue with the {product_purchase...,0.580254


## Knowledge Base

In [24]:
import pandas as pd
from pathlib import Path
knowledge_base_path = Path("../Datasets/knowledge_base")
knowledge_base_path.mkdir(parents=True, exist_ok=True)
print("Knowledge Base folder:", knowledge_base_path)

Knowledge Base folder: ..\Datasets\knowledge_base


#### Create the knowledge documents

In [25]:
knowledge_documents = [

    # Service Policies
    {
        "document_id": "SP001",
        "category": "service_policies",
        "title": "Service Availability",
        "text": "Customer support services are available during the company's standard support hours."
    },
    {
        "document_id": "SP002",
        "category": "service_policies",
        "title": "Service Requests",
        "text": "Customers can contact the support team for questions related to their services and products."
    },
    {
        "document_id": "SP003",
        "category": "service_policies",
        "title": "Service Quality",
        "text": "Support requests should be handled according to the company's service quality standards."
    },

    # Customer Support FAQs
    {
        "document_id": "FAQ001",
        "category": "support_faqs",
        "title": "How can I contact support?",
        "text": "Customers can contact support through the approved customer support channels."
    },
    {
        "document_id": "FAQ002",
        "category": "support_faqs",
        "title": "What information should I provide?",
        "text": "Customers should provide a clear description of their issue and relevant account or product information."
    },
    {
        "document_id": "FAQ003",
        "category": "support_faqs",
        "title": "How is my issue handled?",
        "text": "Support requests are reviewed based on the issue type and assigned support priority."
    },

    # Contract Information
    {
        "document_id": "CON001",
        "category": "contract_information",
        "title": "Contract Duration",
        "text": "The contract duration depends on the service plan selected by the customer."
    },
    {
        "document_id": "CON002",
        "category": "contract_information",
        "title": "Contract Terms",
        "text": "Customers should review the applicable contract terms before making changes to their service."
    },

    # Cancellation Policies
    {
        "document_id": "CAN001",
        "category": "cancellation_policies",
        "title": "Cancellation Request",
        "text": "Customers can submit a cancellation request through the approved customer support process."
    },
    {
        "document_id": "CAN002",
        "category": "cancellation_policies",
        "title": "Cancellation Review",
        "text": "Cancellation requests are reviewed according to the customer's contract and applicable business rules."
    },
    {
        "document_id": "CAN003",
        "category": "cancellation_policies",
        "title": "Cancellation Confirmation",
        "text": "Customers should receive confirmation after their cancellation request has been processed."
    },

    # Billing Policies
    {
        "document_id": "BIL001",
        "category": "billing_policies",
        "title": "Billing Questions",
        "text": "Customers with billing questions should provide relevant billing information so the issue can be investigated."
    },
    {
        "document_id": "BIL002",
        "category": "billing_policies",
        "title": "Payment Issues",
        "text": "Payment issues should be reviewed against the customer's account and available payment records."
    },
    {
        "document_id": "BIL003",
        "category": "billing_policies",
        "title": "Billing Review",
        "text": "Billing concerns should be investigated before any account-related billing action is taken."
    },

    # Retention Guidelines
    {
        "document_id": "RET001",
        "category": "retention_guidelines",
        "title": "Understanding Customer Concerns",
        "text": "Support representatives should understand the customer's concern before discussing retention options."
    },
    {
        "document_id": "RET002",
        "category": "retention_guidelines",
        "title": "Retention Options",
        "text": "Retention options should follow the company's approved retention guidelines."
    },

    # Support Procedures
    {
        "document_id": "SUP001",
        "category": "support_procedures",
        "title": "Ticket Creation",
        "text": "A support ticket should contain enough information to understand and investigate the customer's issue."
    },
    {
        "document_id": "SUP002",
        "category": "support_procedures",
        "title": "Troubleshooting",
        "text": "Support representatives should follow the approved troubleshooting procedure for technical issues."
    },
    {
        "document_id": "SUP003",
        "category": "support_procedures",
        "title": "Issue Escalation",
        "text": "Issues that cannot be resolved through normal support procedures should be escalated to the appropriate support team."
    },

    # Business Rules
    {
        "document_id": "BR001",
        "category": "business_rules",
        "title": "Customer Verification",
        "text": "Customer information should be verified before performing account-specific actions."
    },
    {
        "document_id": "BR002",
        "category": "business_rules",
        "title": "Customer Data Handling",
        "text": "Customer information should be handled according to the company's approved data-handling rules."
    }
]

In [26]:
knowledge_df = pd.DataFrame(knowledge_documents)
print("Number of documents:", len(knowledge_df))
knowledge_df.head()

Number of documents: 21


,document_id,category,title,text
0,SP001,service_policies,Service Availability,Customer support services are available during...
1,SP002,service_policies,Service Requests,Customers can contact the support team for que...
2,SP003,service_policies,Service Quality,Support requests should be handled according t...
3,FAQ001,support_faqs,How can I contact support?,Customers can contact support through the appr...
4,FAQ002,support_faqs,What information should I provide?,Customers should provide a clear description o...


In [27]:
print(knowledge_df["category"].value_counts())

category
service_policies         3
support_faqs             3
cancellation_policies    3
billing_policies         3
support_procedures       3
contract_information     2
retention_guidelines     2
business_rules           2
Name: count, dtype: int64


In [28]:
knowledge_file = knowledge_base_path / "knowledge_base_documents.csv"

knowledge_df.to_csv(
    knowledge_file,
    index=False
)

print("Saved:", knowledge_file)

Saved: ..\Datasets\knowledge_base\knowledge_base_documents.csv


#### Create Knowledge Base chunks

In [29]:
knowledge_chunks_df = knowledge_df.copy()
knowledge_chunks_df["chunk_id"] = [
    f"CH{i:03d}" for i in range(1, len(knowledge_chunks_df) + 1)
]
knowledge_chunks_df = knowledge_chunks_df[
    ["chunk_id", "document_id", "category", "title", "text"]
]
print("Total knowledge chunks:", len(knowledge_chunks_df))

Total knowledge chunks: 21


In [30]:
print(knowledge_chunks_df["category"].value_counts())

category
service_policies         3
support_faqs             3
cancellation_policies    3
billing_policies         3
support_procedures       3
contract_information     2
retention_guidelines     2
business_rules           2
Name: count, dtype: int64


In [31]:
knowledge_chunks_df.head(10)

,chunk_id,document_id,category,title,text
0,CH001,SP001,service_policies,Service Availability,Customer support services are available during...
1,CH002,SP002,service_policies,Service Requests,Customers can contact the support team for que...
2,CH003,SP003,service_policies,Service Quality,Support requests should be handled according t...
3,CH004,FAQ001,support_faqs,How can I contact support?,Customers can contact support through the appr...
4,CH005,FAQ002,support_faqs,What information should I provide?,Customers should provide a clear description o...
5,CH006,FAQ003,support_faqs,How is my issue handled?,Support requests are reviewed based on the iss...
6,CH007,CON001,contract_information,Contract Duration,The contract duration depends on the service p...
7,CH008,CON002,contract_information,Contract Terms,Customers should review the applicable contrac...
8,CH009,CAN001,cancellation_policies,Cancellation Request,Customers can submit a cancellation request th...
9,CH010,CAN002,cancellation_policies,Cancellation Review,Cancellation requests are reviewed according t...


In [32]:
chunks_file = knowledge_base_path / "knowledge_base_chunks.csv"
knowledge_chunks_df.to_csv(
    chunks_file,
    index=False
)
print("Knowledge chunks saved:", chunks_file)

Knowledge chunks saved: ..\Datasets\knowledge_base\knowledge_base_chunks.csv


In [33]:
more_knowledge = [

    # ---------------- Service Policies ----------------
    {
        "document_id": "SP004",
        "category": "service_policies",
        "title": "Service Eligibility",
        "text": "Customers should meet the eligibility requirements associated with the selected service."
    },
    {
        "document_id": "SP005",
        "category": "service_policies",
        "title": "Service Activation",
        "text": "Service activation should follow the standard activation process defined for the selected service."
    },
    {
        "document_id": "SP006",
        "category": "service_policies",
        "title": "Service Changes",
        "text": "Requests to change a service should be reviewed before the requested change is applied."
    },
    {
        "document_id": "SP007",
        "category": "service_policies",
        "title": "Service Requests Priority",
        "text": "Service requests should be prioritized according to their urgency and business priority."
    },
    {
        "document_id": "SP008",
        "category": "service_policies",
        "title": "Service Notifications",
        "text": "Customers should be informed about important service changes or planned interruptions when applicable."
    },
    {
        "document_id": "SP009",
        "category": "service_policies",
        "title": "Service Limitations",
        "text": "Certain services may have limitations based on the selected plan and applicable service conditions."
    },
    {
        "document_id": "SP010",
        "category": "service_policies",
        "title": "Service Requests Review",
        "text": "Requests requiring additional verification should be reviewed before a final action is taken."
    },
    {
        "document_id": "SP011",
        "category": "service_policies",
        "title": "Service Continuity",
        "text": "Support teams should work to minimize service disruption and restore affected services as efficiently as possible."
    },
    {
        "document_id": "SP012",
        "category": "service_policies",
        "title": "Service Communication",
        "text": "Important service-related communication should be provided through approved customer communication channels."
    },
    {
        "document_id": "SP013",
        "category": "service_policies",
        "title": "Service Support Scope",
        "text": "Customer support should handle issues that fall within the approved service support scope."
    },
    {
        "document_id": "SP014",
        "category": "service_policies",
        "title": "Service Request Tracking",
        "text": "Service requests should be tracked until the requested action or resolution has been completed."
    },

    # ---------------- Support FAQs ----------------
    {
        "document_id": "FAQ004",
        "category": "support_faqs",
        "title": "How do I report a product problem?",
        "text": "Customers should describe the product problem and provide relevant details when contacting support."
    },
    {
        "document_id": "FAQ005",
        "category": "support_faqs",
        "title": "What happens after I submit a ticket?",
        "text": "The support team reviews the ticket and handles it according to its issue type and priority."
    },
    {
        "document_id": "FAQ006",
        "category": "support_faqs",
        "title": "Can I update my support request?",
        "text": "Customers may provide additional information to support when further details are required."
    },
    {
        "document_id": "FAQ007",
        "category": "support_faqs",
        "title": "How is ticket priority determined?",
        "text": "Ticket priority is determined according to the urgency and impact of the reported issue."
    },
    {
        "document_id": "FAQ008",
        "category": "support_faqs",
        "title": "What if my issue is not resolved?",
        "text": "Customers can request further assistance when an issue remains unresolved after normal troubleshooting."
    },
    {
        "document_id": "FAQ009",
        "category": "support_faqs",
        "title": "Can technical issues be escalated?",
        "text": "Technical issues that cannot be resolved through normal procedures may be escalated."
    },
    {
        "document_id": "FAQ010",
        "category": "support_faqs",
        "title": "How do I ask about billing?",
        "text": "Customers can contact support with their billing question and relevant account information."
    },
    {
        "document_id": "FAQ011",
        "category": "support_faqs",
        "title": "How do I request cancellation?",
        "text": "Customers should submit a cancellation request through the approved support process."
    },
    {
        "document_id": "FAQ012",
        "category": "support_faqs",
        "title": "Can support check account information?",
        "text": "Support may review account information after the required customer verification is completed."
    },
    {
        "document_id": "FAQ013",
        "category": "support_faqs",
        "title": "What should I include in a support ticket?",
        "text": "A support ticket should include the issue description, relevant product or account details, and useful troubleshooting information."
    },
    {
        "document_id": "FAQ014",
        "category": "support_faqs",
        "title": "How can I follow up?",
        "text": "Customers can follow up through the approved support channel and provide their existing ticket information."
    },
    {
        "document_id": "FAQ015",
        "category": "support_faqs",
        "title": "Can support help with account access?",
        "text": "Support can assist with account access issues after completing the required verification process."
    },
    {
        "document_id": "FAQ016",
        "category": "support_faqs",
        "title": "What if I entered incorrect information?",
        "text": "Customers should contact support to request correction of incorrect account or service information."
    },
    {
        "document_id": "FAQ017",
        "category": "support_faqs",
        "title": "Can I ask for a status update?",
        "text": "Customers can request an update on an existing support request through the approved support channel."
    },
    {
        "document_id": "FAQ018",
        "category": "support_faqs",
        "title": "What if I have several issues?",
        "text": "Customers should clearly describe each issue so that support can determine the appropriate handling process."
    },

    # ---------------- Contract Information ----------------
    {
        "document_id": "CON003",
        "category": "contract_information",
        "title": "Contract Start",
        "text": "The contract becomes effective according to the activation or start terms associated with the selected service."
    },
    {
        "document_id": "CON004",
        "category": "contract_information",
        "title": "Contract Renewal",
        "text": "Contract renewal should follow the renewal terms associated with the customer's agreement."
    },
    {
        "document_id": "CON005",
        "category": "contract_information",
        "title": "Contract Cancellation Terms",
        "text": "Cancellation requests are subject to the terms and conditions stated in the customer's contract."
    },
    {
        "document_id": "CON006",
        "category": "contract_information",
        "title": "Contract Changes",
        "text": "Changes to contract-related information should follow the applicable contract modification process."
    },
    {
        "document_id": "CON007",
        "category": "contract_information",
        "title": "Contract Responsibilities",
        "text": "Customers and service providers should follow the responsibilities defined by the applicable contract."
    },
    {
        "document_id": "CON008",
        "category": "contract_information",
        "title": "Contract Review",
        "text": "Customers should review important contract conditions before requesting account or service changes."
    },
    {
        "document_id": "CON009",
        "category": "contract_information",
        "title": "Contract Eligibility",
        "text": "Contract options may depend on the customer's selected service and eligibility conditions."
    },
    {
        "document_id": "CON010",
        "category": "contract_information",
        "title": "Contract Information Requests",
        "text": "Requests for contract information should be handled after verifying the customer's account."
    },
    {
        "document_id": "CON011",
        "category": "contract_information",
        "title": "Contract Exceptions",
        "text": "Contract exceptions should be reviewed according to approved business procedures."
    },
    {
        "document_id": "CON012",
        "category": "contract_information",
        "title": "Contract Records",
        "text": "Relevant contract information should be maintained in the appropriate customer records."
    },

    # ---------------- Cancellation Policies ----------------
    {
        "document_id": "CAN004",
        "category": "cancellation_policies",
        "title": "Cancellation Eligibility",
        "text": "Cancellation eligibility depends on the customer's service agreement and applicable cancellation conditions."
    },
    {
        "document_id": "CAN005",
        "category": "cancellation_policies",
        "title": "Cancellation Processing",
        "text": "Cancellation requests should be processed using the approved cancellation procedure."
    },
    {
        "document_id": "CAN006",
        "category": "cancellation_policies",
        "title": "Cancellation Verification",
        "text": "Customer identity and account information should be verified before processing cancellation."
    },
    {
        "document_id": "CAN007",
        "category": "cancellation_policies",
        "title": "Cancellation Status",
        "text": "Customers may request the current status of their cancellation request through support."
    },
    {
        "document_id": "CAN008",
        "category": "cancellation_policies",
        "title": "Cancellation Documentation",
        "text": "Cancellation requests should contain sufficient information for support to identify the correct account and service."
    },
    {
        "document_id": "CAN009",
        "category": "cancellation_policies",
        "title": "Cancellation Escalation",
        "text": "Cancellation issues that cannot be resolved through the normal process should be escalated."
    },
    {
        "document_id": "CAN010",
        "category": "cancellation_policies",
        "title": "Cancellation and Billing",
        "text": "Billing-related conditions associated with cancellation should be reviewed before final processing."
    },
    {
        "document_id": "CAN011",
        "category": "cancellation_policies",
        "title": "Cancellation Communication",
        "text": "Customers should receive appropriate communication regarding the status of their cancellation request."
    },
    {
        "document_id": "CAN012",
        "category": "cancellation_policies",
        "title": "Cancellation Record",
        "text": "Completed cancellation requests should be recorded in the appropriate customer account records."
    },

    # ---------------- Billing Policies ----------------
    {
        "document_id": "BIL004",
        "category": "billing_policies",
        "title": "Billing Verification",
        "text": "Customer identity and relevant account information should be verified before discussing account-specific billing details."
    },
    {
        "document_id": "BIL005",
        "category": "billing_policies",
        "title": "Billing Dispute",
        "text": "Billing disputes should be reviewed against the available account and billing records."
    },
    {
        "document_id": "BIL006",
        "category": "billing_policies",
        "title": "Billing Corrections",
        "text": "Billing corrections should be made only after the issue has been reviewed and verified."
    },
    {
        "document_id": "BIL007",
        "category": "billing_policies",
        "title": "Payment Method Issues",
        "text": "Payment method problems should be investigated using the relevant account and payment information."
    },
    {
        "document_id": "BIL008",
        "category": "billing_policies",
        "title": "Billing Information Requests",
        "text": "Requests for billing information should follow the customer verification procedure."
    },
    {
        "document_id": "BIL009",
        "category": "billing_policies",
        "title": "Billing Escalation",
        "text": "Complex billing issues may be escalated to the appropriate billing support team."
    },
    {
        "document_id": "BIL010",
        "category": "billing_policies",
        "title": "Billing Records",
        "text": "Relevant billing information should be maintained in the appropriate customer records."
    },
    {
        "document_id": "BIL011",
        "category": "billing_policies",
        "title": "Billing Communication",
        "text": "Customers should receive clear communication regarding investigated billing issues."
    },
    {
        "document_id": "BIL012",
        "category": "billing_policies",
        "title": "Billing Review Process",
        "text": "Billing concerns should be reviewed systematically before a final decision is made."
    },

    # ---------------- Retention Guidelines ----------------
    {
        "document_id": "RET003",
        "category": "retention_guidelines",
        "title": "Identify Customer Concerns",
        "text": "Retention discussions should begin by identifying the customer's primary concern."
    },
    {
        "document_id": "RET004",
        "category": "retention_guidelines",
        "title": "Offer Appropriate Support",
        "text": "Support representatives should offer retention options that are appropriate for the customer's situation."
    },
    {
        "document_id": "RET005",
        "category": "retention_guidelines",
        "title": "Respect Customer Decisions",
        "text": "Customer retention activities should respect the customer's final decision regarding the service."
    },
    {
        "document_id": "RET006",
        "category": "retention_guidelines",
        "title": "Retention Escalation",
        "text": "Complex retention cases may be escalated according to the applicable support process."
    },
    {
        "document_id": "RET007",
        "category": "retention_guidelines",
        "title": "Retention Documentation",
        "text": "Important retention interactions should be recorded in the appropriate customer support records."
    },
    {
        "document_id": "RET008",
        "category": "retention_guidelines",
        "title": "Retention and Service Quality",
        "text": "Retention activities should focus on solving customer concerns and maintaining service quality."
    },
    {
        "document_id": "RET009",
        "category": "retention_guidelines",
        "title": "Retention Follow Up",
        "text": "Follow-up may be used when further action is required to address a customer's concern."
    },
    {
        "document_id": "RET010",
        "category": "retention_guidelines",
        "title": "Retention Guidelines Review",
        "text": "Retention actions should remain consistent with approved company guidelines."
    },

    # ---------------- Support Procedures ----------------
    {
        "document_id": "SUP004",
        "category": "support_procedures",
        "title": "Ticket Verification",
        "text": "Support representatives should review the ticket information before starting investigation."
    },
    {
        "document_id": "SUP005",
        "category": "support_procedures",
        "title": "Ticket Categorization",
        "text": "Tickets should be categorized according to the type of customer issue."
    },
    {
        "document_id": "SUP006",
        "category": "support_procedures",
        "title": "Priority Assignment",
        "text": "Tickets should receive an appropriate priority based on urgency and customer impact."
    },
    {
        "document_id": "SUP007",
        "category": "support_procedures",
        "title": "Initial Investigation",
        "text": "Support representatives should perform an initial investigation before recommending further action."
    },
    {
        "document_id": "SUP008",
        "category": "support_procedures",
        "title": "Troubleshooting Documentation",
        "text": "Important troubleshooting steps should be recorded in the support ticket."
    },
    {
        "document_id": "SUP009",
        "category": "support_procedures",
        "title": "Escalation Criteria",
        "text": "Issues that exceed normal support capabilities should be escalated according to procedure."
    },
    {
        "document_id": "SUP010",
        "category": "support_procedures",
        "title": "Resolution Recording",
        "text": "The resolution or recommended action should be recorded after an issue has been handled."
    },
    {
        "document_id": "SUP011",
        "category": "support_procedures",
        "title": "Customer Follow Up",
        "text": "Support representatives should follow up when additional customer information is required."
    },
    {
        "document_id": "SUP012",
        "category": "support_procedures",
        "title": "Unresolved Tickets",
        "text": "Unresolved tickets should remain active until the required next step has been completed."
    },

    # ---------------- Business Rules ----------------
    {
        "document_id": "BR003",
        "category": "business_rules",
        "title": "Account Access",
        "text": "Account-specific actions should only be performed after the required customer verification."
    },
    {
        "document_id": "BR004",
        "category": "business_rules",
        "title": "Customer Privacy",
        "text": "Customer information should only be used and shared according to approved business rules."
    },
    {
        "document_id": "BR005",
        "category": "business_rules",
        "title": "Support Authorization",
        "text": "Only authorized support personnel should perform restricted account actions."
    },
    {
        "document_id": "BR006",
        "category": "business_rules",
        "title": "Data Accuracy",
        "text": "Customer and service records should be kept accurate and updated when corrections are verified."
    },
    {
        "document_id": "BR007",
        "category": "business_rules",
        "title": "Issue Escalation Rule",
        "text": "Issues requiring specialized handling should be escalated to the appropriate team."
    },
    {
        "document_id": "BR008",
        "category": "business_rules",
        "title": "Customer Communication",
        "text": "Customer communication should use approved support channels and appropriate language."
    },
    {
        "document_id": "BR009",
        "category": "business_rules",
        "title": "Record Keeping",
        "text": "Important support actions and decisions should be recorded in the appropriate system."
    },
    {
        "document_id": "BR010",
        "category": "business_rules",
        "title": "Priority Handling",
        "text": "High-priority customer issues should receive appropriate attention according to support rules."
    },
    {
        "document_id": "BR011",
        "category": "business_rules",
        "title": "Information Verification",
        "text": "Important customer information should be verified before account-specific decisions are made."
    },
    {
        "document_id": "BR012",
        "category": "business_rules",
        "title": "Policy Compliance",
        "text": "Support actions should follow the applicable company policies and business rules."
    },
]

In [34]:
more_knowledge_df = pd.DataFrame(more_knowledge)
knowledge_df = pd.concat([knowledge_df, more_knowledge_df],ignore_index=True)
print("Total knowledge documents:", len(knowledge_df))

Total knowledge documents: 102


In [35]:
knowledge_chunks_df = knowledge_df.copy()

knowledge_chunks_df["chunk_id"] = [
    f"CH{i:03d}" for i in range(1, len(knowledge_chunks_df) + 1)
]

knowledge_chunks_df = knowledge_chunks_df[
    ["chunk_id", "document_id", "category", "title", "text"]
]

print("Total knowledge chunks:", len(knowledge_chunks_df))

Total knowledge chunks: 102


In [36]:
print("Total documents:", len(knowledge_df))
print()
print(knowledge_df["category"].value_counts())

Total documents: 102

category
support_faqs             18
service_policies         14
contract_information     12
cancellation_policies    12
billing_policies         12
support_procedures       12
business_rules           12
retention_guidelines     10
Name: count, dtype: int64


In [37]:
knowledge_df.tail()

,document_id,category,title,text
97,BR008,business_rules,Customer Communication,Customer communication should use approved sup...
98,BR009,business_rules,Record Keeping,Important support actions and decisions should...
99,BR010,business_rules,Priority Handling,High-priority customer issues should receive a...
100,BR011,business_rules,Information Verification,Important customer information should be verif...
101,BR012,business_rules,Policy Compliance,Support actions should follow the applicable c...


In [38]:
print("Number of documents:", len(knowledge_df))
print("Number of unique document IDs:", knowledge_df["document_id"].nunique())

Number of documents: 102
Number of unique document IDs: 102


In [39]:
chunks_file = knowledge_base_path / "knowledge_base_chunks.csv"
knowledge_chunks_df.to_csv(chunks_file,index=False)
print("Knowledge Base saved:", chunks_file)

Knowledge Base saved: ..\Datasets\knowledge_base\knowledge_base_chunks.csv


In [40]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded
Embedding dimension: 384


C:\Users\Admin\AppData\Local\Temp\ipykernel_13400\762695290.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [41]:
knowledge_texts = knowledge_chunks_df["text"].tolist()
print("Number of texts:", len(knowledge_texts))

Number of texts: 102


#### Create the embeddings

In [42]:
knowledge_embeddings = embedding_model.encode(
    knowledge_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", knowledge_embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding shape: (102, 384)


In [43]:
from sklearn.preprocessing import normalize
import numpy as np
knowledge_embeddings_normalized = normalize(knowledge_embeddings,norm="l2").astype("float32")
print(
    "Normalized embedding shape:",
    knowledge_embeddings_normalized.shape
)

Normalized embedding shape: (102, 384)


In [44]:
embedding_file = "../Datasets/processed/knowledge_base_embeddings.npy"
np.save(embedding_file,knowledge_embeddings_normalized)
print("Knowledge Base embeddings saved:", embedding_file)

Knowledge Base embeddings saved: ../Datasets/processed/knowledge_base_embeddings.npy


In [45]:
print("Chunks:", len(knowledge_chunks_df))
print("Embeddings:", len(knowledge_embeddings_normalized))
print("Dimensions:", knowledge_embeddings_normalized.shape[1])

Chunks: 102
Embeddings: 102
Dimensions: 384


#### Create Knowledge Base FAISS index

In [46]:
embedding_dimension = knowledge_embeddings_normalized.shape[1]
print("Embedding dimension:", embedding_dimension)

Embedding dimension: 384


#### Create the FAISS index

In [47]:
knowledge_base_index = faiss.IndexFlatIP(embedding_dimension)

In [48]:
#add knowledge base embeddings
knowledge_base_index.add(knowledge_embeddings_normalized)

In [49]:
print("Number of vectors:", knowledge_base_index.ntotal)

Number of vectors: 102


In [50]:
knowledge_faiss_file = "../Datasets/processed/knowledge_base_faiss.index"
faiss.write_index(knowledge_base_index,knowledge_faiss_file)
print("Knowledge Base FAISS index saved:", knowledge_faiss_file)

Knowledge Base FAISS index saved: ../Datasets/processed/knowledge_base_faiss.index


In [51]:
print("Knowledge chunks :", len(knowledge_chunks_df))
print("Embeddings       :", len(knowledge_embeddings_normalized))
print("FAISS vectors    :", knowledge_base_index.ntotal)
print("Embedding dim    :", knowledge_embeddings_normalized.shape[1])

Knowledge chunks : 102
Embeddings       : 102
FAISS vectors    : 102
Embedding dim    : 384


In [52]:
def search_knowledge_base(query, top_k=5):   
    # Convert the user query into an embedding
    query_embedding = embedding_model.encode([query],convert_to_numpy=True)
    # Normalize the query embedding
    query_embedding = normalize(query_embedding,norm="l2").astype("float32")
    # Search the Knowledge Base FAISS index
    distances, indices = knowledge_base_index.search(query_embedding,top_k)
    # Get the matching knowledge chunks
    results = knowledge_chunks_df.iloc[indices[0]].copy()
    # Add similarity score
    results["similarity_score"] = distances[0]
    # Return clean result
    return results.reset_index(drop=True)

In [53]:
results = search_knowledge_base("How can I cancel my service?")
results

,chunk_id,document_id,category,title,text,similarity_score
0,CH040,FAQ011,support_faqs,How do I request cancellation?,Customers should submit a cancellation request...,0.702715
1,CH009,CAN001,cancellation_policies,Cancellation Request,Customers can submit a cancellation request th...,0.652583
2,CH059,CAN005,cancellation_policies,Cancellation Processing,Cancellation requests should be processed usin...,0.597142
3,CH061,CAN007,cancellation_policies,Cancellation Status,Customers may request the current status of th...,0.594546
4,CH058,CAN004,cancellation_policies,Cancellation Eligibility,Cancellation eligibility depends on the custom...,0.578919


In [54]:
for i, row in results.iterrows():
    print(f"Rank: {i + 1}")
    print(f"Similarity: {row['similarity_score']:.4f}")
    print(f"Category: {row['category']}")
    print(f"Title: {row['title']}")
    print(f"Text: {row['text']}")
    print("-" * 60)

Rank: 1
Similarity: 0.7027
Category: support_faqs
Title: How do I request cancellation?
Text: Customers should submit a cancellation request through the approved support process.
------------------------------------------------------------
Rank: 2
Similarity: 0.6526
Category: cancellation_policies
Title: Cancellation Request
Text: Customers can submit a cancellation request through the approved customer support process.
------------------------------------------------------------
Rank: 3
Similarity: 0.5971
Category: cancellation_policies
Title: Cancellation Processing
Text: Cancellation requests should be processed using the approved cancellation procedure.
------------------------------------------------------------
Rank: 4
Similarity: 0.5945
Category: cancellation_policies
Title: Cancellation Status
Text: Customers may request the current status of their cancellation request through support.
------------------------------------------------------------
Rank: 5
Similarity: 0.5789
Categ